# Устанавливаем зависимости

In [2]:
%pip install transformer_lens
%pip install einops
%pip install jaxtyping
%pip install git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/callummcdougall/CircuitsVis.git to c:\users\владимир\appdata\local\temp\pip-req-build-to8fai7o
Note: you may need to restart the kernel to use updated packages.


  ERROR: Error [WinError 2] Не удается найти указанный файл while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


In [ ]:
import os; os.environ['ACCELERATE_DISABLE_RICH'] = "1"
from dataclasses import dataclass
from transformer_lens import HookedTransformer
import torch as t
import torch
from torch import Tensor
import torch.nn as nn
import numpy as np
import math
from tqdm.notebook import tqdm
from jaxtyping import Float, Int
from transformers.models.gpt2.tokenization_gpt2_fast import GPT2TokenizerFast
from collections import defaultdict

device = t.device("cuda" if t.cuda.is_available() else "cpu")

reference_gpt2 = HookedTransformer.from_pretrained("gpt2-small", fold_ln=False, center_unembed=False, center_writing_weights=False)
reference_gpt2 = reference_gpt2.to(device)

c:\Users\Владимир\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu


Конфиг, который хранит в себе всю информацию о размерностях модели.

In [4]:
@dataclass
class Config:
    d_model: int = 768 # он же hidden_dim - внутрення размерность модели
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257 # он же vocab_size, размер словаря модели
    init_range: float = 0.02
    n_ctx: int = 1024 # число позиционных эмбеддингов
    d_head: int = 64 # размерность головы аттеншена
    d_mlp: int = 3072 # внутренняя размерность FFN-слоя
    n_heads: int = 12 # число голов аттеншена
    n_layers: int = 12 # число слоев трансформера

cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


Тесты для проверки размерностей

In [5]:
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randn(shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple): output = output[0]
    print("Output shape:", output.shape, "\n")

def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randint(100, 1000, shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple): output = output[0]
    print("Output shape:", output.shape, "\n")

def load_gpt2_test(cls, gpt2_layer, input):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    print("Input shape:", input.shape)
    output = layer(input)
    if isinstance(output, tuple): output = output[0]
    print("Output shape:", output.shape)
    try: reference_output = gpt2_layer(input)
    except: reference_output = gpt2_layer(input, input, input)
    print("Reference output shape:", reference_output.shape, "\n")
    comparison = t.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of the values are correct\n")

In [6]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text).to(device)
print(tokens)
print(tokens.shape)
print(reference_gpt2.to_str_tokens(tokens))

tensor([[50256,    40,   716,   281,  4998,  1960,   382, 19741,    11,   875,
         12342,    12,  8807,    11,   402, 11571,    12,    17,  3918, 47385,
            13,  1881,  1110,   314,   481,  7074,  1692,  1241,  4430,   290,
          1011,   625,   262,   995,     0]])
torch.Size([1, 35])
['<|endoftext|>', 'I', ' am', ' an', ' amazing', ' aut', 'ore', 'gressive', ',', ' dec', 'oder', '-', 'only', ',', ' G', 'PT', '-', '2', ' style', ' transformer', '.', ' One', ' day', ' I', ' will', ' exceed', ' human', ' level', ' intelligence', ' and', ' take', ' over', ' the', ' world', '!']


In [ ]:
logits, cache = reference_gpt2.run_with_cache(tokens)
print(logits.shape)

torch.Size([1, 35, 50257])
Все работает, мы готовы к выполнению задания!


# Embeddings

`[batch_size, seq_len]` -> `[batch_size, seq_len, d_model]`

In [8]:
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model)))  
        nn.init.normal_(self.W_E, std=self.cfg.init_range)

    def forward(self, input_ids: Int[Tensor, "batch seq_len"]) -> Float[Tensor, "batch seq_len d_model"]:
        return torch.nn.functional.embedding(input_ids, self.W_E)


batch_size = 2
seq_len = 4
rand_int_test(Embed, [batch_size, seq_len])
load_gpt2_test(Embed, reference_gpt2.embed, tokens)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35])
Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



# Position Embeddings

1. По tokens получить тензор positions размера `[batch_size, seq_len]`
2. Заэмбеддить тензор positions, как в предыдущем слое.


In [9]:
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, input_ids: Int[Tensor, "batch seq_len"]) -> Float[Tensor, "batch seq_len d_model"]:
        batch, seq_len = input_ids.shape
        positions = torch.arange(0, seq_len).unsqueeze(0)
        positions = positions.repeat(batch, 1)
        return torch.nn.functional.embedding(positions, self.W_pos)

batch_size = 2
seq_len = 4
rand_int_test(PosEmbed, [batch_size, seq_len])
load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35])
Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



# LM head

Финальный слой. У нас есть выходы из трансформера размерности `[batch_size, seq_len, d_model]`.  
По ним мы предсказываем следующий токен, т.е. применяем линейный слой - умножаем на матрицу `[d_model, vocab_size]`.


In [ ]:
# LM_head, для совместимости с библиотекой для проверки пришлось назвать его Unembed
# по аналогии с тем, что мы из индексов в словаре получаем эмбеддинги, а тут из эмбеддингов обратно
# распределение по словарю

class Unembed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(t.empty((cfg.d_model, cfg.d_vocab)))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(t.zeros((cfg.d_vocab), requires_grad=False))

    def forward(
        self, x: Float[Tensor, "batch seq_len d_model"]) -> Float[Tensor, "batch seq_len d_vocab"]:
        return torch.nn.functional.linear(x, self.W_U.T, self.b_U)


batch_size = 2
seq_len = 4
d_model = 768
# rand_float_test(Unembed, [batch_size, seq_len, d_model])
load_gpt2_test(Unembed, reference_gpt2.unembed, cache["ln_final.hook_normalized"])

Input shape: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 50257])
Reference output shape: torch.Size([1, 35, 50257]) 

100.00% of the values are correct



# Attention

1. Нам попадает на вход вектор x `[batch, seq_len, d_model]`. Нужно превратить его в матрицы проекций i-й головы аттеншена: Q_i, K_i, V_i. Для этого у нас есть матрицы W_Q, W_K, W_V. Это набор n_heads матриц размеров `[d_model, d_head]` (d_model == n_head * d_head)  Нужно перевести `[num_heads, d_model, d_head]` в матрицу `[d_model, num_heads * d_head]` = `[d_model, d_model]`, после чего получить `[batch_size, seq_len, d_model]`, получить матрицы Q, K, V размерностей `[batch_size, seq_len, d_model] = [batch_size, seq_len, num_heads * d_head]` и преобразовать их к виду `[batch_size, seq_len, num_heads, d_head]`.

2. Вычислить attention_scores.

3. Нормализация

4. Маскирование (masked attention).

5. Softmax по dim=-1.

6. Матричное умножение softmax(attention_scores) на V

7. Конкатенировать в исходную размерность.


In [ ]:
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))

        self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))

        self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))

        self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        self.b_O = nn.Parameter(t.zeros((cfg.d_model)))

        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def forward(
        self, x: Float[Tensor, "batch seq_len d_model"]
    ) -> Float[Tensor, "batch seq_len d_model"]:

        # Берем размерности
        batch_size, seq_len, d_model = x.shape
        num_heads = self.cfg.n_heads
        d_head = self.cfg.d_head
        # 1. Трансформируем матрицы проекций в формат [d_model, d_model]
        W_Q = self.W_Q.permute(1, 0, 2).reshape(self.cfg.d_model, self.cfg.d_model)
        W_K = self.W_K.permute(1, 0, 2).reshape(self.cfg.d_model, self.cfg.d_model)
        W_V = self.W_V.permute(1, 0, 2).reshape(self.cfg.d_model, self.cfg.d_model)

        b_Q = self.b_Q.view(-1)
        b_K = self.b_K.view(-1)
        b_V = self.b_V.view(-1)

        # 1. получаем проекции  Q, K, V
        Q = (torch.matmul(x, W_Q) + b_Q).view(batch_size, seq_len, num_heads, d_head)

        K = torch.nn.functional.linear(x, W_K.T, b_K).view(batch_size, seq_len, num_heads, d_head)
        V = torch.nn.functional.linear(x, W_V.T, b_V).view(batch_size, seq_len, num_heads, d_head)
        # 2. Q x K^T
        attention_scores = torch.matmul(Q.permute(0, 2, 1, 3), K.permute(0, 2, 3, 1))
        # 3. Нормализация
        attention_scores = attention_scores / self.cfg.d_head ** 0.5
        # 4. Маскирование
        attention_scores_masked = self.apply_causal_mask(attention_scores)
        # print(attention_scores_masked)
        # 5. softmax
        attn_probs = torch.softmax(attention_scores_masked, dim=-1)

        # 6. Финальная проекция
        # V - [batch, seq_len, num_heads, d_head]
        # permute [ batch, num_heads, seq_len, d_head]
        output = torch.matmul(attn_probs, V.permute(0, 2, 1, 3))
        output = output.permute(0, 2, 1, 3).reshape(batch_size, seq_len, num_heads * d_head)
        W_O = self.W_O.view(d_model, d_model)
        b_O = self.b_O.view(-1)
        res = nn.functional.linear(output, W_O.T, b_O)
        return res

    def apply_causal_mask(
        self, attn_scores: Float[Tensor, "batch n_heads seq_len seq_len"]
    ) -> Float[Tensor, "batch n_heads seq_len seq_len"]:
        '''
        Используем треугольную маску, чтобы не смотреть в будущее
        '''
        mask = torch.tril(torch.ones_like(attn_scores)).bool()
        return attn_scores.masked_fill(~mask, self.IGNORE)

torch.manual_seed(1)
batch_size = 2
seq_len = 4
d_model = 768

# rand_float_test(Attention, [batch_size, seq_len, d_model])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["normalized", 0, "ln1"])


Input shape: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



In [12]:
attn = Attention(cfg)

In [ ]:
import json
data = json.load(open("attention_tensors.json"))
x = torch.Tensor(data["x"])
# qjson = torch.Tensor(data["Q"])

In [ ]:
W_Q = attn.W_Q.permute(1, 0, 2).reshape(cfg.d_model, cfg.d_model)
b_Q = attn.b_Q.view(-1)

Ql = torch.nn.functional.linear(x, W_Q.T, b_Q).view(batch_size, seq_len, cfg.n_heads, -1)
Qm = (torch.matmul(x, W_Q) + b_Q).view(batch_size, seq_len, cfg.n_heads, -1)


print(torch.allclose(Ql, Qm, atol=1e-3, rtol=1e-4))

True


In [ ]:
(Ql == Qm).all()

tensor(True)

In [ ]:
Q = torch.nn.functional.linear

# MLP и Unembedding

Реализуем MLP слой - это 2 матричных умножения с нелинейностью GELU.


In [14]:
class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(t.empty((cfg.d_model, cfg.d_mlp)))
        self.W_out = nn.Parameter(t.empty((cfg.d_mlp, cfg.d_model)))
        self.b_in = nn.Parameter(t.zeros((cfg.d_mlp)))
        self.b_out = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        nn.init.normal_(self.W_out, std=self.cfg.init_range)

    def forward(
        self, x: Float[Tensor, "batch seq_len d_model"]
    ) -> Float[Tensor, "batch seq_len d_model"]:
        x2 = torch.nn.functional.linear(x, self.W_in.T, self.b_in)
        x3 = torch.nn.functional.gelu(x2, approximate="tanh")
        out = torch.nn.functional.linear(x3, self.W_out.T, self.b_out)
        return out

torch.manual_seed(1)

rand_float_test(MLP, [batch_size, seq_len, d_model])
load_gpt2_test(MLP, reference_gpt2.blocks[0].mlp, cache["normalized", 0, "ln2"])

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



# Normalization

**Layer Normalization**
   

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(t.ones(cfg.d_model)) # gamma
        self.b = nn.Parameter(t.zeros(cfg.d_model)) # beta

    def forward(self, x: Float[Tensor, "batch seq_len d_model"]) -> Float[Tensor, "batch seq_len d_model"]:
        m = x.mean(dim=-1, keepdim=True)
        sigma = (x.var(dim=-1, keepdim=True, unbiased=False) + self.cfg.layer_norm_eps) ** 0.5
        return (x - m) / sigma * self.w + self.b



rand_float_test(LayerNorm, [2, 4, 768])
load_gpt2_test(LayerNorm, reference_gpt2.ln_final, cache["resid_post", 11])

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



# Full transformer

Это блок трансформера, который получает на вход тензор x `[batch_size, seq_len, d_model]` и выдает тензор таких же размерностей. 

Следуем схеме PreLN

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)

    def forward(
        self, x: Float[Tensor, "batch seq_len d_model"]
    ) -> Float[Tensor, "batch seq_len d_model"]:
        attn = self.attn(self.ln1(x))
        x = x + attn
        return x + self.mlp(self.ln2(x))


rand_float_test(TransformerBlock, [2, 4, 768])
load_gpt2_test(TransformerBlock, reference_gpt2.blocks[0], cache["resid_pre", 0])

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 35, 768])
Output shape: torch.Size([1, 35, 768])
Reference output shape: torch.Size([1, 35, 768]) 

100.00% of the values are correct



Собираем все в один большой трансформер.
1. Применяем эмбеддинги и позиционные эмбеддинги, складываем результаты
2. Прогоняем в цикле через все блоки трансформера
3. Применяем финальную нормализацию и lm_head

In [ ]:
class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)

    def forward(self, input_ids: Int[Tensor, "batch seq_len"]) -> Float[Tensor, "batch seq_len d_vocab"]:

        x = self.embed(input_ids) + self.pos_embed(input_ids)
        for layer in self.blocks:
            x = layer(x)

        return self.unembed(self.ln_final(x))



rand_int_test(DemoTransformer, [2, 4])
load_gpt2_test(DemoTransformer, reference_gpt2, tokens)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 50257]) 

Input shape: torch.Size([1, 35])
Output shape: torch.Size([1, 35, 50257])
Reference output shape: torch.Size([1, 35, 50257]) 

100.00% of the values are correct



In [ ]:
demo_gpt2 = DemoTransformer(Config(debug=False)).to(device)
demo_gpt2.load_state_dict(reference_gpt2.state_dict(), strict=False)

demo_logits = demo_gpt2(tokens)

In [ ]:
demo_gpt2

DemoTransformer(
  (embed): Embed()
  (pos_embed): PosEmbed()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNorm()
      (attn): Attention()
      (ln2): LayerNorm()
      (mlp): MLP()
    )
  )
  (ln_final): LayerNorm()
  (unembed): Unembed()
)

In [ ]:
reference_gpt2

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (hook_re

In [ ]:
def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"],
    tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:

    log_probs = logits.log_softmax(dim=-1)
    log_probs_for_tokens = log_probs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)

    return log_probs_for_tokens


pred_log_probs = get_log_probs(demo_logits, tokens)
print(f"Avg cross entropy loss: {-pred_log_probs.mean():.4f}")
print(f"Avg cross entropy loss for uniform distribution: {math.log(demo_gpt2.cfg.d_vocab):4f}")
print(f"Avg probability assigned to correct token: {pred_log_probs.exp().mean():4f}")

Avg cross entropy loss: 4.5647
Avg cross entropy loss for uniform distribution: 10.824905
Avg probability assigned to correct token: 0.087911


In [ ]:
test_string = '''The Total Perspective Vortex derives its picture of the whole Universe on the principle of'''
for i in tqdm(range(100)):
    test_tokens = reference_gpt2.to_tokens(test_string).to(device)
    demo_logits = demo_gpt2(test_tokens)
    test_string += reference_gpt2.tokenizer.decode(demo_logits[-1, -1].argmax())

print(test_string)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/100 [00:00<?, ?it/s]

The Total Perspective Vortex derives its picture of the whole Universe on the principle of the total perspective. The total perspective is the view of the whole Universe from the point of view of the observer. The total perspective is the view of the whole Universe from the point of view of the observer. The total perspective is the view of the whole Universe from the point of view of the observer. The total perspective is the view of the whole Universe from the point of view of the observer. The total perspective is the view of the whole Universe from the point of view of the observer. The


# Sampling


1. **Temperature Sampling**

2. **Frequency Penalty**

3. **Top-k Sampling**

4. **Top-p (Nucleus Sampling)**

In [ ]:
model_cfg = Config()
model = DemoTransformer(model_cfg).to(device)
model.load_state_dict(reference_gpt2.state_dict(), strict=False)

tokenizer = reference_gpt2.tokenizer

In [1]:
class TransformerSampler:

    def __init__(self, model: DemoTransformer, tokenizer: GPT2TokenizerFast):
        self.model = model
        self.cfg = model.cfg
        self.tokenizer = tokenizer

    @t.inference_mode()
    def sample(self, prompt: str, max_tokens_generated=100, verbose=False, **kwargs):
        if prompt == '':
            prompt = '<|endoftext|>'
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").flatten().to(device)
        input_ids = input_ids.view(1, -1)

        for _ in range(max_tokens_generated):
            logits = self.model(input_ids)
            logits = logits[:, -1, :]
            next_id = self.sample_next_token(input_ids[0], logits, **kwargs)
            if next_id > self.cfg.d_vocab:
                break
            next_id = t.tensor([next_id]).to(device).unsqueeze(0)
            input_ids = t.cat((input_ids, next_id), 1)

        out = ''
        for idx in input_ids:
            out += tokenizer.decode(idx.cpu().squeeze())
        return out



    @staticmethod
    def sample_next_token(input_ids: Int[Tensor, "seq_len"],
        logits: Float[Tensor, "d_vocab"],
        temperature=1.0,
        top_k=0,
        top_p=0.0,
        frequency_penalty=0.0,
        seed=None):
        assert input_ids.ndim == 1
        assert temperature >= 0
        assert 0 <= top_p <= 1.0
        assert 0 <= top_k
        assert not (top_p != 0 and top_k != 0)

        if seed is not None:
            t.manual_seed(seed)
            np.random.seed(seed)

        if temperature == 0:
            return TransformerSampler.greedy_search(logits)
        elif temperature != 1.0:
            logits = TransformerSampler.apply_temperature(logits, temperature)
        if frequency_penalty != 0.0:
            logits = TransformerSampler.apply_frequency_penalty(input_ids, logits, frequency_penalty)
        if top_k > 0:
            return TransformerSampler.sample_top_k(logits, top_k)
        if top_p > 0.0:
            return TransformerSampler.sample_top_p(logits, top_p)
        return TransformerSampler.sample_basic(logits)


    @staticmethod
    def greedy_search(logits: Float[Tensor, "d_vocab"]) -> int:
        out = logits.argmax().item()
        return out


    @staticmethod
    def apply_temperature(logits: Float[Tensor, "d_vocab"], temperature: float) -> Float[Tensor, "d_vocab"]:
        return logits / temperature


    @staticmethod
    def apply_frequency_penalty(input_ids: Int[Tensor, "seq_len"], logits: Float[Tensor, "d_vocab"], freq_penalty: float) -> Float[Tensor, "d_vocab"]:
        token_counts = torch.bincount(input_ids, minlength=logits.size(-1))
        penalty = freq_penalty * token_counts.float()
        penalized_logits = logits - penalty
        return penalized_logits


    @staticmethod
    def sample_basic(logits: Float[Tensor, "d_vocab"]) -> int:
        probs = torch.softmax(logits, dim=-1)
        return torch.multinomial(probs, 1).item()


    @staticmethod
    def sample_top_k(logits: Float[Tensor, "d_vocab"], k: int) -> int:
        probs = torch.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(logits, k)
        topk_probs = topk_probs / topk_probs.sum()
        topk_word = torch.multinomial(topk_probs, 1).item()
        original_index = topk_indices[topk_word]
        return logits[original_index].item()


    @staticmethod
    def sample_top_p(logits: Float[Tensor, "d_vocab"], top_p: float, min_tokens_to_keep: int = 1) -> int:
        logits_sorted, indices = logits.sort(descending=True, stable=True)
        cumul_probs = logits_sorted.softmax(-1).cumsum(-1)
        n_keep = t.searchsorted(cumul_probs, top_p, side="left").item() + 1
        n_keep = max(n_keep, min_tokens_to_keep)
        keep_idx = indices[:n_keep]
        keep_logits = logits[keep_idx]
        sample = t.distributions.categorical.Categorical(logits=keep_logits).sample()
        return keep_idx[sample].item()

NameError: name 'DemoTransformer' is not defined

In [ ]:
sampler = TransformerSampler(model, tokenizer)

prompt = "Jingle bells, jingle bells, jingle all the way"
print(f"Greedy decoding with prompt: {prompt!r}\n")

output = sampler.sample(prompt, max_tokens_generated=8, temperature=0.0)
print(f"Your model said: {output!r}\n")

expected = "Jingle bells, jingle bells, jingle all the way up to the top of the mountain."
assert output == expected

print("Tests passed!")

Greedy decoding with prompt: 'Jingle bells, jingle bells, jingle all the way'

Your model said: 'Jingle bells, jingle bells, jingle all the way up to the top of the mountain.'

Tests passed!


In [ ]:
logits = t.tensor([1, 2]).log()

cold_logits = TransformerSampler.apply_temperature(logits, temperature=0.001)
print('A low temperature "sharpens" or "peaks" the distribution: ', cold_logits)
t.testing.assert_close(cold_logits, 1000.0 * logits)

hot_logits = TransformerSampler.apply_temperature(logits, temperature=1000.0)
print("A high temperature flattens the distribution: ", hot_logits)
t.testing.assert_close(hot_logits, 0.001 * logits)

print("Tests passed!")

A low temperature "sharpens" or "peaks" the distribution:  tensor([  0.0000, 693.1472])
A high temperature flattens the distribution:  tensor([0.0000, 0.0007])
Tests passed!


In [ ]:
bieber_prompt = "And I was like Baby, baby, baby, oh Like, Baby, baby, baby, no Like, Baby, baby, baby, oh I thought you'd always be mine, mine"
input_ids = tokenizer.encode(bieber_prompt, return_tensors="pt")
logits = t.ones(tokenizer.vocab_size)
penalized_logits = TransformerSampler.apply_frequency_penalty(input_ids.squeeze(), logits, 2.0)

assert penalized_logits[5156].item() == -11, "Expected 6 occurrences of ' baby' with leading space, 1-2*6=-11"
assert penalized_logits[14801].item() == -5, "Expected 3 occurrences of ' Baby' with leading space, 1-2*3=-5"

print("Tests passed!")

Tests passed!


In [ ]:
prompt = "John and Mary went to the"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
logits = model(input_ids)[0, -1]

expected_top_10pct = {
    " church": 0.0648,
    " house": 0.0367, # These are the two most likely tokens, and add up to >10%
}
top_10pct_sum = sum(expected_top_10pct.values())

observed_freqs = defaultdict(int)

N = 10000
for _ in tqdm(range(N)):
    token = TransformerSampler.sample_next_token(input_ids.squeeze(), logits, top_p=0.1)
    observed_freqs[tokenizer.decode(token)] += 1

for word in expected_top_10pct:
    expected_freq = expected_top_10pct[word] / top_10pct_sum
    observed_freq = observed_freqs[word] / N
    print(f"Word: {word!r:<9}. Expected freq {expected_freq:.4f}, observed freq {observed_freq:.4f}")
    assert abs(observed_freq - expected_freq) < 0.01, "Try increasing N if this fails by a small amount."

  0%|          | 0/10000 [00:00<?, ?it/s]

tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.

tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.

tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.

tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.0648, 0.1015, 0.1160,  ..., 1.0000, 1.0000, 1.0000],
       grad_fn=<CumsumBackward0>)
2
tensor([0.

KeyboardInterrupt: 